# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook walks through loading, exploring, and processing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
# Print name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their IDs, and get a preview of the records for each record set.

**Note:** All entities are referenced by their `@id`.

In [ ]:
# List record sets defined in the dataset
record_sets = list(dataset.record_sets.keys())
print("Available record sets (by @id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# For each record set, list fields (by @id)
for rs_id in record_sets:
    print(f"\nFields for record set {rs_id}:")
    fields = dataset.record_sets[rs_id].fields
    for f in fields:
        print(f"  - {f['@id']}: {f.get('name', f['@id'])}")

    # Preview: Print first 2 records
    print(f"Preview of first 2 records in {rs_id}:")
    for idx, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if idx >= 1:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from all record sets

dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame columns for record set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head(3))
    else:
        print(f"Record set {record_set_id} is empty.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations shown: filtering, normalization, grouping by attributes.

We will pick the first available record set with data and demo some analyses.

In [ ]:
# Pick a record set with data
record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        record_set_id = rs_id
        break
if record_set_id is None:
    raise RuntimeError("No record sets with data found.")

df = dataframes[record_set_id]

# Identify numeric fields (columns) via heuristics (example: field names containing 'Age', 'Interval', 'TumorSize', etc.)
numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'size' in col.lower()]

if not numeric_fields:
    # If heuristics fail, pick first column with numeric dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
            break

if numeric_fields:
    # Use first numeric field
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    # Apply a threshold (e.g., > 10) if meaningful
    try:
        threshold = 10
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()

        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field (e.g., 'Sex' or 'MSI Status')
        group_fields = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouped field: {group_field}")
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            )
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No group field found for grouping.")
    except Exception as e:
        print(f"Could not filter or normalize numeric field: {e}")
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize numeric field distributions and relationships between attributes.

We will create a histogram (distribution) of the selected numeric field and, if possible, a boxplot grouped by a chosen categorical field.

In [ ]:
# Visualization
if record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    plt.hist(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=10, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by a categorical field
    if group_fields:
        plt.figure(figsize=(8, 5))
        # Drop rows with NA
        boxplot_data = df[[numeric_field, group_field]].copy()
        boxplot_data[numeric_field] = pd.to_numeric(boxplot_data[numeric_field], errors='coerce')
        boxplot_data = boxplot_data.dropna()
        boxplot_data.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze a Croissant-based dataset using `mlcroissant` by referencing entities through their `@id` fields.

- **Data loaded successfully:** The dataset provides valuable clinical and molecular variables for colorectal cancer survivors.
- **Record sets and fields explored:** IDs were used throughout for robust referencing.
- **Basic EDA performed:** Filtering, normalizing, and grouping demonstrate how the clinical data can be processed.
- **Visualizations:** Numeric distributions and relationships highlight potential insights for biomarker-driven oncology research.

Further exploration could include correlation analyses, cross-record set joins, or machine learning applications.
